# LLM API Explorer

> *Understanding how Large Language Models work at the API level — before any abstraction, framework, or shortcut.*

---

## What is this project?

Most people jump straight into ChatGPT wrappers, LangChain, or AI tools without understanding what's happening underneath. This project fixes that.

You will talk **directly to the Gemini API**, control every parameter manually, and observe exactly how each change affects the model's output — building the mental model that every serious AI project depends on.

---

## Project Details

| | |
|---|---|
| **API Used** | Google Gemini 2.5 Flash (free tier) |
| **Language** | Python 3 |
| **Environment** | Jupyter Notebook via Anaconda |

---

## What you will build

| Phase | What happens |
|---|---|
| **Phase 2** | First API call — connect to Gemini, send a message, read the reply, understand token usage |
| **Phase 3** | Experiment with system prompts, roles, temperature, and top-p — same question, completely different answers |
| **Phase 4** | Count tokens *before* sending using `tiktoken` — understand context windows and why limits matter |
| **Phase 5** | Build a logging system — every call saved with its settings, reply, and token count to a CSV file |
| **Phase 6** | Build an evaluation harness — compare outputs side by side, score them, draw conclusions |

---

## Skills you will have after this project

- ✅ Understand the **role system** — system / user / assistant (model)
- ✅ Know what **temperature** and **top-p** actually do to output
- ✅ **Count tokens before sending** — cost and context awareness
- ✅ **Log and compare** LLM outputs systematically
- ✅ Build a basic **evaluation framework** from scratch
- ✅ **Debug LLM behaviour** confidently in any future project

---

## Stack

```
google-genai      — Gemini API SDK
tiktoken          — Token counting
pandas            — Logging and comparison tables  
python-dotenv     — Secure API key management
```

---

> **Key idea:** Every AI application ever built — no matter how complex — is doing exactly one thing at its core: sending text in and getting text out. Everything else is just organising that loop better. This project makes that loop completely transparent.

# Phase 1 
### Store Your API Key Safely & Make Your First API Call

# Check if notebook and .env are in the same folder

In [1]:
# Check working directory and find .env file
# WHAT THIS CELL DOES:
# Jupyter runs from a "home" folder. Your .env file must be in
# the SAME folder as your notebook. This cell confirms that.

import os  # 'os' is a built-in Python tool to interact with your computer's file system

# os.getcwd() = "get current working directory" = where Jupyter is currently running from
current_folder = os.getcwd()

print("Jupyter is currently running from this folder:")
print(current_folder)
print()

# os.path.exists() checks if a file exists at the given location
# ".env" means "look for .env in the current folder"
if os.path.exists(".env"):
    print("✅ .env file found! Your API key file is in the right place.")
else:
    print("❌ .env file NOT found.")
    print()
    print("👉 Your notebook is running from:", current_folder)
    print("👉 Your .env file is saved in:   C:\\Users\\ishaa\\OneDrive\\Documents\\Projects\\LLM API Explorer")
    print()
    print("These two must match. Either:")
    print("   1. Move your .env file to:", current_folder)
    print("   2. Or open Jupyter directly from your Projects folder")

Jupyter is currently running from this folder:
C:\Users\ishaa\OneDrive\Documents\Projects\LLM API Explorer

✅ .env file found! Your API key file is in the right place.


# Load your API key and connect to Gemini

In [2]:
# Load API key from .env file and connect to Gemini
# WHAT THIS CELL DOES:
# 1. Reads your secret API key from the .env file
# 2. Uses that key to "log in" to Gemini
# 3. Creates a 'client' object — your connection to Gemini
#
# WHY .env FILE?
# If you paste the key directly in code and share the notebook,
# everyone sees your key. .env keeps it separate and secret.

import os                    # to read environment variables (key-value pairs from .env)
from dotenv import load_dotenv  # to load the .env file into Python's memory
from google import genai     # the official Google Gemini SDK (library)

# Step 1: Load the .env file 
# load_dotenv() reads your .env file and loads GEMINI_API_KEY
# into Python's environment so os.getenv() can find it
load_dotenv()

# Step 2: Read the key 
# os.getenv("GEMINI_API_KEY") looks for a variable named
# GEMINI_API_KEY inside the loaded .env file and returns its value
api_key = os.getenv("GEMINI_API_KEY")

# Step 3: Confirm the key loaded correctly 
if api_key:
    print("✅ API key loaded successfully!")
    # We print only the first 8 and last 4 characters for safety
    # so you can verify it's correct without exposing the full key
    print(f"   Key preview: {api_key[:8]}...{api_key[-4:]}")
else:
    print("❌ API key not found.")
    print("   Check that your .env file contains exactly:")
    print("   GEMINI_API_KEY=your_actual_key_here")
    print("   (no spaces around the = sign)")

print()

# Step 4: Create the Gemini client 
# genai.Client() creates a connection object using your API key
# Think of it like logging into a website — once logged in,
# you can make as many requests as you want using this 'client'
client = genai.Client(api_key=api_key)

print("✅ Gemini client created! You are now connected to Gemini.")
print("   You can now send messages to the AI.")

✅ API key loaded successfully!
   Key preview: AQ.Ab8RN...IrTA

✅ Gemini client created! You are now connected to Gemini.
   You can now send messages to the AI.


# First API Call with gemini-2.5-flash
The reply got cut off mid-sentence ("and perform many other language..."). That's because 300 tokens wasn't enough for a complete answer.

`total_token_count = 311` even though you set `max_output_tokens = 300`. That's because the limit applies only to the reply, not your question. Your question itself used 15 tokens, so 15 + 60 = 75... the rest (236) are "thinking tokens" — gemini-2.5-flash thinks silently before answering.

In [3]:
# Your first API call to Gemini
# WHAT THIS CELL DOES:
# Sends a question to Gemini and reads the reply.
#
# NEW CONCEPTS HERE:
# 1. gemini-2.5-flash → latest free Gemini model, smarter than 2.0
# 2. max_output_tokens → limits how long the reply can be
#
# WHAT IS A TOKEN?
# A token is NOT a word. It's a chunk of text.
# Rule of thumb: 1 word ≈ 1.3 tokens
# "Hello" = 1 token | "Artificial Intelligence" = 3 tokens
# Why it matters: APIs charge by tokens, and models have a
# maximum number of tokens they can read+write at once (context window)
# gemini-2.5-flash context window = 1,000,000 tokens (huge)


from google.genai import types  # 'types' lets us set advanced options like token limits

# Step 1: Send the message 
response = client.models.generate_content(
    model="gemini-2.5-flash",           # latest free Gemini model

    contents="What is a large language model? Explain in 3 simple sentences.",

    config=types.GenerateContentConfig(
        max_output_tokens=300,          # reply cannot exceed 300 tokens (~200 words)
                                        # if you set this too low, reply gets cut off mid-sentence
                                        # for short answers: 100-300 | essays: 1000+ | no limit: don't set this
    )
)

# Step 2: Print the reply
print("You asked:")
print("What is a large language model? Explain in 3 simple sentences.")
print()
print("Gemini replied:")
print("─" * 60)
print(response.text)
print("─" * 60)
print()

# Step 3: Print token usage 
# usage_metadata tells you exactly how many tokens were consumed
# prompt_token_count     = tokens in YOUR message (the question)
# candidates_token_count = tokens in GEMINI'S reply
# total_token_count      = both combined

usage = response.usage_metadata
print("📊 Token Usage:")
print(f"   Your message used       : {usage.prompt_token_count} tokens")
print(f"   Gemini's reply used     : {usage.candidates_token_count} tokens")
print(f"   Total tokens consumed   : {usage.total_token_count} tokens")
print()
print(f"   Max output you allowed  : 300 tokens")
print(f"   Tokens remaining unused : {300 - usage.candidates_token_count} tokens")

You asked:
What is a large language model? Explain in 3 simple sentences.

Gemini replied:
────────────────────────────────────────────────────────────
A Large Language Model (LLM) is an advanced computer program designed to understand and generate human language. It learns by processing vast amounts of text data, allowing it to recognize patterns, context, and grammar. This enables it to answer questions, write stories, translate languages, and perform many other language
────────────────────────────────────────────────────────────

📊 Token Usage:
   Your message used       : 15 tokens
   Gemini's reply used     : 60 tokens
   Total tokens consumed   : 311 tokens

   Max output you allowed  : 300 tokens
   Tokens remaining unused : 240 tokens


# Phase 2 - System Prompts, Roles, Temperature, Top-P

## The Core Idea First

Every message you send to an LLM has a role attached to it:

| Role | Who it is | What it does |
|------|------|------|
| `system` | You (the developer) | Sets the AI's personality, rules, and behaviour before the conversation starts |
| `user` | The person talking | The actual question or message |
| `assistant` | The AI | Its reply — you can also pre-fill this to guide responses |

---

Temperature controls randomness:

- `0.0` = robotic, deterministic, same answer every time
- `1.0` = balanced, natural
- `2.0` = creative, unpredictable, sometimes weird

---

Top-p controls vocabulary diversity:

- `0.1` = picks only from the most likely words (safe, repetitive)
- `0.9` = picks from a wide range of words (varied, creative)

# Same Question, Three Different System Prompts

In [5]:
# System Prompts: Same question, completely different answers
# WHAT THIS CELL DOES:
# Sends the exact same user question 3 times.
# Only the system prompt changes each time.
# You will see how dramatically the reply changes.
#
# WHAT IS A SYSTEM PROMPT?
# It's a secret instruction you give the AI before the
# conversation starts. The user never sees it.
# It defines WHO the AI is and HOW it should behave.
# Example: "You are a strict professor" vs "You are a friendly tutor"
# — same question, completely different tone and depth of answer.
# 
# WHY 503 HAPPENS:
# Free tier has rate limits — too many requests too fast = server
# refuses. Adding a small delay between calls prevents this.

from google.genai import types
import time   # built-in Python library — lets us pause execution

user_question = "What is machine learning?"

system_prompts = {

    "👶 Simple (Explain to a 10-year-old)":
        "You are a friendly teacher explaining things to a 10-year-old child. "
        "Use very simple words, short sentences, and a fun analogy.",

    "🎓 Expert (Talk to a data scientist)":
        "You are a senior ML researcher. The user is a data scientist. "
        "Be technical, precise, and concise. Skip basic definitions.",

    "😄 Funny (Stand-up comedian)":
        "You are a stand-up comedian. Answer every question with humour, "
        "sarcasm, and jokes — but still make sure the answer is correct.",
}

for i, (label, system_prompt) in enumerate(system_prompts.items()):

    print(f"\n{'=' * 60}")
    print(f"SYSTEM PROMPT: {label}")
    print(f"{'=' * 60}")

    # Retry logic — if 503 happens, wait and try again automatically
    max_retries = 3        # try up to 3 times before giving up
    retry_delay = 15       # wait 15 seconds between retries

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    {"role": "user", "parts": [{"text": user_question}]}
                ],
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    max_output_tokens=400,   # increased — enough for a complete answer
                    temperature=1.0,
                )
            )

            print(f"\n🤖 Reply:\n{response.text}")
            print(f"\n📊 Tokens used: {response.usage_metadata.total_token_count}")
            break   # success — exit the retry loop

        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"   ⚠️  503 error on attempt {attempt+1}. Waiting {retry_delay}s and retrying...")
                time.sleep(retry_delay)   # wait before retrying
            else:
                print(f"   ❌ Unexpected error: {e}")
                break   # not a 503 — don't retry

    # Wait between each system prompt call (not just retries)
    # Only wait if there's a next call coming
    if i < len(system_prompts) - 1:
        print(f"\n   ⏳ Waiting 10 seconds before next call...")
        time.sleep(10)   # pause between calls to respect free tier limits

print("\n✅ All 3 system prompt experiments complete!")


SYSTEM PROMPT: 👶 Simple (Explain to a 10-year-old)

🤖 Reply:
Imagine you have a super-duper smart robot friend!



📊 Tokens used: 433

   ⏳ Waiting 10 seconds before next call...

SYSTEM PROMPT: 🎓 Expert (Talk to a data scientist)
   ⚠️  503 error on attempt 1. Waiting 15s and retrying...

🤖 Reply:
Machine learning is a computational paradigm where systems learn patterns and make inferences from data,

📊 Tokens used: 429

   ⏳ Waiting 10 seconds before next call...

SYSTEM PROMPT: 😄 Funny (Stand-up comedian)

🤖 Reply:
Alright, alright, settle down folks! You want to know about machine

📊 Tokens used: 431

✅ All 3 system prompt experiments complete!


# Temperature Experiment

In [6]:
# CELL 5 — Temperature: How randomness changes everything
# WHAT THIS CELL DOES:
# Sends the exact same question 4 times with different temperatures.
# Everything else stays identical.
# Watch how the reply changes from robotic → creative → chaotic.
#
# TEMPERATURE EXPLAINED:
# Imagine the AI is choosing the next word from a bag of options.
# temperature=0.0 → always picks the single most likely word (boring but safe)
# temperature=1.0 → picks naturally, like a human would write
# temperature=2.0 → picks randomly from unlikely words too (creative or nonsense)
#
# WHEN TO USE WHAT:
# Low  (0.0-0.3) → factual Q&A, code generation, data extraction
# Mid  (0.7-1.0) → conversation, explanation, summaries
# High (1.5-2.0) → brainstorming, creative writing, story generation

from google.genai import types
import time

user_question = "Give me a one-sentence description of what a neural network is."
system_prompt = "You are a helpful AI assistant."
temperatures = [0.0, 0.7, 1.5, 2.0]

print("EXPERIMENT: Same question, same system prompt, different temperatures")
print(f"Question: {user_question}\n")

for i, temp in enumerate(temperatures):

    max_retries = 3
    retry_delay = 15

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    {"role": "user", "parts": [{"text": user_question}]}
                ],
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    max_output_tokens=500,    # enough for a complete one-liner + thinking tokens
                    temperature=temp,         # ← only this changes
                )
            )

            print(f"🌡️  Temperature = {temp}")
            print(f"   {response.text.strip()}")
            print(f"   Tokens used: {response.usage_metadata.total_token_count}")
            print()
            break

        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"   ⚠️  503 on attempt {attempt+1}. Waiting {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"   ❌ Unexpected error: {e}")
                break

    if i < len(temperatures) - 1:
        print(f"   ⏳ Waiting 10 seconds before next call...\n")
        time.sleep(10)

print("✅ Temperature experiment complete!")

EXPERIMENT: Same question, same system prompt, different temperatures
Question: Give me a one-sentence description of what a neural network is.

🌡️  Temperature = 0.0
   A neural network is a computational system, loosely modeled on the human brain, that learns from data to recognize patterns and make predictions or decisions.
   Tokens used: 497

   ⏳ Waiting 10 seconds before next call...

🌡️  Temperature = 0.7
   A neural network is a computational model, inspired by the human brain's structure, that learns patterns from data to make predictions or classifications.
   Tokens used: 84

   ⏳ Waiting 10 seconds before next call...

🌡️  Temperature = 1.5
   A neural network is a computational model inspired by the human brain that learns from data to recognize patterns, make predictions, or classify information without explicit programming for each task.
   Tokens used: 95

   ⏳ Waiting 10 seconds before next call...

🌡️  Temperature = 2.0
   A neural network is a computational model, l

# The Role System (Multi-turn Conversation)

In [7]:
# ============================================================
# CELL 6 — Roles: user / assistant / system in a real conversation
# ============================================================
# WHAT THIS CELL DOES:
# Shows how a multi-turn conversation works at the API level.
# IMPORTANT: Gemini has NO memory between calls.
# If you want it to remember earlier messages, YOU must
# send the full conversation history every single time.
# This is how every chatbot works under the hood.
#
# CONVERSATION STRUCTURE:
# [system]    → sets the AI's behaviour (sent via system_instruction)
# [user]      → your message
# [assistant] → AI's reply (included in history for next turn)
# [user]      → your next message
# [assistant] → AI's reply again
# ... and so on
# ============================================================

from google.genai import types

system_prompt = (
    "You are a data science mentor helping a fresher MBA student "
    "learn machine learning concepts from scratch. "
    "Be encouraging, clear, and use simple real-world examples."
)

# This is the full conversation history — you build it turn by turn
# Each message is a dict with 'role' and 'parts'
conversation_history = [
    {"role": "user",      "parts": [{"text": "Hi! I just started learning ML. Where do I begin?"}]},
    {"role": "model",     "parts": [{"text": "Great place to start! Begin with understanding what ML actually is — teaching computers to learn from examples instead of following fixed rules. Start with supervised learning: give the model labelled data, and it learns to predict. Want me to walk you through your first concept?"}]},
    {"role": "user",      "parts": [{"text": "Yes please. What is supervised learning exactly?"}]},
]

# NOTE: in Gemini's API, assistant role is called "model" not "assistant"
# OpenAI uses "assistant" — same concept, different label

print("Conversation so far:")
print("─" * 60)
for msg in conversation_history:
    role_label = "👤 You" if msg["role"] == "user" else "🤖 Gemini"
    print(f"{role_label}: {msg['parts'][0]['text']}")
print("─" * 60)

# Send the full history — Gemini reads ALL of it to understand context
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=conversation_history,      # full history, not just the last message
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=800,
        temperature=0.7,
    )
)

print(f"\nGemini (Turn 3 reply):")
print(response.text)
print()
print(f"📊 Total tokens for this full conversation: {response.usage_metadata.total_token_count}")
print()
print("💡 KEY INSIGHT:")
print("   Notice total tokens is HIGH — because you sent ALL 3 turns.")
print("   Every turn you add makes the next call more expensive.")
print("   This is why long conversations cost more tokens over time.")

Conversation so far:
────────────────────────────────────────────────────────────
👤 You: Hi! I just started learning ML. Where do I begin?
🤖 Gemini: Great place to start! Begin with understanding what ML actually is — teaching computers to learn from examples instead of following fixed rules. Start with supervised learning: give the model labelled data, and it learns to predict. Want me to walk you through your first concept?
👤 You: Yes please. What is supervised learning exactly?
────────────────────────────────────────────────────────────

Gemini (Turn 3 reply):
Okay, fantastic! Let's dive into Supervised Learning. It's one of the most common and intuitive types of machine learning, so it's a perfect place to start.

Imagine you're teaching a child to identify different fruits.

*   You show them an **apple** and say, "This is an **apple**."
*   You show them a **banana** and say, "This is a **banana**."
*   You show them an **orange** and say, "This is an **orange**."

You do this m

# Phase 4 - Token Counting BEFORE Sending
Right now you only see token counts after the API responds. In real projects you need to count tokens before sending — to avoid hitting context limits or overspending.
`tiktoken` is OpenAI's tokenizer but works as a close approximation for Gemini too. Good enough for learning the concept.

In [8]:
# Token Counting Before Sending (Phase 4)
# WHAT THIS CELL DOES:
# Shows you how to estimate token count BEFORE making an API call.
# This is critical in real projects where:
#   - You have context window limits (max tokens model can handle)
#   - You want to estimate cost before sending
#   - You want to truncate long inputs automatically
#
# TWO WAYS TO COUNT:
# 1. tiktoken  → local, instant, free, works offline (approximation)
# 2. Gemini's own counter → exact, but uses one API call itself
#
# We use tiktoken to save your daily API quota.
#
# HOW TOKENIZATION WORKS:
# Text gets split into chunks called tokens before the model reads it.
# "machine learning" → ["machine", " learning"] → 2 tokens
# "ChatGPT"          → ["Chat", "G", "PT"]      → 3 tokens
# Numbers, punctuation, spaces all count as tokens too.


import tiktoken

# Load the tokenizer
# "cl100k_base" is the encoding used by GPT-4 — close approximation for Gemini
encoder = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    """Count how many tokens a piece of text uses."""
    tokens = encoder.encode(text)   # converts text into a list of token integers
    return len(tokens)              # number of tokens = length of that list

def estimate_call(system_prompt, user_message, max_output_tokens):
    """
    Estimates total tokens for a planned API call BEFORE sending it.
    Shows whether you're within safe limits.
    """
    system_tokens = count_tokens(system_prompt)
    user_tokens   = count_tokens(user_message)
    input_tokens  = system_tokens + user_tokens

    print(f"📊 Token Estimate (BEFORE sending):")
    print(f"   System prompt tokens : {system_tokens}")
    print(f"   User message tokens  : {user_tokens}")
    print(f"   Total input tokens   : {input_tokens}")
    print(f"   Max output allowed   : {max_output_tokens}")
    print(f"   Worst-case total     : {input_tokens + max_output_tokens}")
    print()

    # Gemini 2.5 Flash context window = 1,000,000 tokens
    context_limit = 1_000_000
    if input_tokens + max_output_tokens > context_limit:
        print("❌ WARNING: Exceeds context window! Shorten your input.")
    else:
        remaining = context_limit - (input_tokens + max_output_tokens)
        print(f"✅ Well within context limit. {remaining:,} tokens still available.")

# Example 1: Short prompt 
print("=" * 60)
print("EXAMPLE 1: Short system prompt + short question")
print("=" * 60)
estimate_call(
    system_prompt    = "You are a helpful assistant.",
    user_message     = "What is machine learning?",
    max_output_tokens= 500
)

# Example 2: Long detailed system prompt 
print("=" * 60)
print("EXAMPLE 2: Detailed system prompt + longer question")
print("=" * 60)
estimate_call(
    system_prompt    = (
        "You are an expert data science mentor helping MBA students "
        "understand machine learning from scratch. Always use real-world "
        "business examples. Break down complex concepts into simple steps. "
        "After every explanation, ask if the student wants to go deeper."
    ),
    user_message     = (
        "Can you explain the difference between supervised and unsupervised "
        "learning, and give me a real-world business example for each?"
    ),
    max_output_tokens= 800
)

# Example 3: Tokenize word by word 
print("=" * 60)
print("EXAMPLE 3: See exactly how text breaks into tokens")
print("=" * 60)

sample_text = "Machine learning models learn patterns from data."

tokens      = encoder.encode(sample_text)         # list of token IDs (numbers)
token_words = [encoder.decode([t]) for t in tokens]  # decode each ID back to text chunk

print(f"Original text : {sample_text}")
print(f"Token count   : {len(tokens)}")
print(f"Tokens        : {token_words}")
print()
print("💡 Notice: spaces are attached to the NEXT word, not the previous one.")
print("   This is how modern tokenizers work.")

EXAMPLE 1: Short system prompt + short question
📊 Token Estimate (BEFORE sending):
   System prompt tokens : 6
   User message tokens  : 5
   Total input tokens   : 11
   Max output allowed   : 500
   Worst-case total     : 511

✅ Well within context limit. 999,489 tokens still available.
EXAMPLE 2: Detailed system prompt + longer question
📊 Token Estimate (BEFORE sending):
   System prompt tokens : 44
   User message tokens  : 24
   Total input tokens   : 68
   Max output allowed   : 800
   Worst-case total     : 868

✅ Well within context limit. 999,132 tokens still available.
EXAMPLE 3: See exactly how text breaks into tokens
Original text : Machine learning models learn patterns from data.
Token count   : 8
Tokens        : ['Machine', ' learning', ' models', ' learn', ' patterns', ' from', ' data', '.']

💡 Notice: spaces are attached to the NEXT word, not the previous one.
   This is how modern tokenizers work.


# Phases 5 & 6 - Combined (Logging + Evaluation)

In [9]:
# Logging System + Evaluation Harness (Phases 5 & 6)
# WHAT THIS CELL DOES:
# Phase 5: Logs every API call to a CSV file automatically
#          (prompt used, settings, reply, token count, timestamp)
# Phase 6: Compares all logged calls side by side and scores them
#
# WHY LOGGING MATTERS:
# When you run 20 experiments, you can't remember which prompt
# gave which output. A log file makes every experiment reproducible
# and comparable — this is standard practice in ML projects.
#
# WHY EVALUATION MATTERS:
# "Which prompt worked best?" needs a systematic answer, not a guess.
# An evaluation harness scores outputs against criteria so you can
# make data-driven decisions about prompt design.

import csv
import datetime
import time
import os
import pandas as pd
from google.genai import types
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")

# PART 1: Setup logging 
LOG_FILE = "llm_experiment_log.csv"   # saved in same folder as your notebook

# CSV column headers — one row per API call
log_columns = [
    "timestamp",          # when the call was made
    "experiment_name",    # label you give this experiment
    "system_prompt",      # system prompt used
    "user_message",       # what you asked
    "temperature",        # temperature setting
    "max_output_tokens",  # token limit set
    "reply",              # what Gemini said
    "input_tokens_est",   # estimated input tokens (tiktoken)
    "total_tokens_used",  # actual total tokens (from API response)
    "reply_score",        # your manual score out of 10
    "score_reason",       # why you gave that score
]

# Create the CSV file with headers if it doesn't exist yet
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=log_columns)
        writer.writeheader()
    print(f"✅ Log file created: {LOG_FILE}")
else:
    print(f"✅ Log file already exists: {LOG_FILE}")

def log_call(experiment_name, system_prompt, user_message,
             temperature, max_output_tokens, reply,
             input_tokens_est, total_tokens_used,
             reply_score, score_reason):
    """Appends one experiment result as a row in the CSV log."""
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=log_columns)
        writer.writerow({
            "timestamp"         : datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "experiment_name"   : experiment_name,
            "system_prompt"     : system_prompt[:80] + "..." if len(system_prompt) > 80 else system_prompt,
            "user_message"      : user_message,
            "temperature"       : temperature,
            "max_output_tokens" : max_output_tokens,
            "reply"             : reply.strip(),
            "input_tokens_est"  : input_tokens_est,
            "total_tokens_used" : total_tokens_used,
            "reply_score"       : reply_score,
            "score_reason"      : score_reason,
        })

# PART 2: Define experiments 
# 3 experiments — 3 API calls total (saves your daily quota)
# Same question, different system prompts, we score and compare

experiments = [
    {
        "name"              : "Formal Expert",
        "system_prompt"     : "You are a formal ML expert. Be precise and technical.",
        "user_message"      : "What is overfitting in machine learning?",
        "temperature"       : 0.3,
        "max_output_tokens" : 500,
        "score_criteria"    : "Is it accurate, clear, and appropriately technical?",
    },
    {
        "name"              : "Friendly Tutor",
        "system_prompt"     : "You are a friendly tutor. Use simple words and a real-life analogy.",
        "user_message"      : "What is overfitting in machine learning?",
        "temperature"       : 0.7,
        "max_output_tokens" : 500,
        "score_criteria"    : "Is the analogy good? Would a beginner understand this?",
    },
    {
        "name"              : "One-Line Summary",
        "system_prompt"     : "You are a concise assistant. Answer in exactly one sentence.",
        "user_message"      : "What is overfitting in machine learning?",
        "temperature"       : 0.0,
        "max_output_tokens" : 500,
        "score_criteria"    : "Is it truly one sentence and still accurate?",
    },
]

# PART 3: Run experiments, log each one 
print("\nRunning 3 experiments...\n")

results = []   # store results in memory for evaluation below

for i, exp in enumerate(experiments):

    print(f"{'=' * 60}")
    print(f"EXPERIMENT {i+1}: {exp['name']}")
    print(f"{'=' * 60}")

    # Count input tokens before sending
    input_tokens_est = (
        len(encoder.encode(exp["system_prompt"])) +
        len(encoder.encode(exp["user_message"]))
    )

    # Retry logic
    max_retries = 3
    reply = None

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    {"role": "user", "parts": [{"text": exp["user_message"]}]}
                ],
                config=types.GenerateContentConfig(
                    system_instruction=exp["system_prompt"],
                    max_output_tokens=exp["max_output_tokens"],
                    temperature=exp["temperature"],
                )
            )
            reply = response.text
            total_tokens = response.usage_metadata.total_token_count
            break

        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"   ⚠️  503 on attempt {attempt+1}. Waiting 15s...")
                time.sleep(15)
            else:
                print(f"   ❌ Error: {e}")
                break

    if not reply:
        print("   ❌ All retries failed. Skipping this experiment.")
        continue

    print(f"\n🤖 Reply:\n{reply}")
    print(f"\n📊 Input tokens (estimated) : {input_tokens_est}")
    print(f"   Total tokens used        : {total_tokens}")

    # Manual scoring 
    # In a real eval harness this would be automated (LLM-as-judge)
    # For now you score it manually based on the criteria
    print(f"\n Score criteria: {exp['score_criteria']}")

    # Auto-score based on simple rules (beginner-friendly version)
    # Real projects use LLM-as-judge or human raters
    word_count = len(reply.split())

    if exp["name"] == "One-Line Summary":
        # Score based on whether it's actually one sentence
        sentence_count = reply.count(".") + reply.count("!") + reply.count("?")
        score  = 9 if sentence_count <= 1 else 6
        reason = "One sentence ✅" if sentence_count <= 1 else f"Multiple sentences ({sentence_count}) ❌"
    elif exp["name"] == "Friendly Tutor":
        # Score based on whether it used an analogy
        has_analogy = any(word in reply.lower() for word in ["like", "imagine", "think of", "similar to", "as if"])
        score  = 9 if has_analogy else 7
        reason = "Contains analogy ✅" if has_analogy else "No clear analogy found"
    else:
        # Formal expert — score based on length (too short = not detailed enough)
        score  = 9 if word_count > 50 else 6
        reason = f"Detailed ({word_count} words) ✅" if word_count > 50 else f"Too brief ({word_count} words)"

    print(f"   Auto-score : {score}/10 — {reason}")

    # Log this experiment
    log_call(
        experiment_name   = exp["name"],
        system_prompt     = exp["system_prompt"],
        user_message      = exp["user_message"],
        temperature       = exp["temperature"],
        max_output_tokens = exp["max_output_tokens"],
        reply             = reply,
        input_tokens_est  = input_tokens_est,
        total_tokens_used = total_tokens,
        reply_score       = score,
        score_reason      = reason,
    )

    results.append({
        "Experiment"  : exp["name"],
        "Temperature" : exp["temperature"],
        "Words"       : word_count,
        "Score"       : score,
        "Reason"      : reason,
    })

    if i < len(experiments) - 1:
        print(f"\n   ⏳ Waiting 10 seconds...\n")
        time.sleep(10)

# PART 4: Evaluation summary table 
print(f"\n{'=' * 60}")
print("📊 EVALUATION SUMMARY (Phase 6)")
print(f"{'=' * 60}\n")

df = pd.DataFrame(results)
print(df.to_string(index=False))

print(f"\n✅ All results saved to: {LOG_FILE}")

✅ Log file created: llm_experiment_log.csv

Running 3 experiments...

EXPERIMENT 1: Formal Expert

🤖 Reply:
Overfitting is a fundamental problem in machine learning where a model learns the training data too specifically,

📊 Input tokens (estimated) : 21
   Total tokens used        : 517

 Score criteria: Is it accurate, clear, and appropriately technical?
   Auto-score : 6/10 — Too brief (17 words)

   ⏳ Waiting 10 seconds...

EXPERIMENT 2: Friendly Tutor

🤖 Reply:
Hey there! That's a super common and important question in machine learning. Let's break

📊 Input tokens (estimated) : 24
   Total tokens used        : 521

 Score criteria: Is the analogy good? Would a beginner understand this?
   Auto-score : 7/10 — No clear analogy found

   ⏳ Waiting 10 seconds...

EXPERIMENT 3: One-Line Summary

🤖 Reply:
Overfitting occurs when a machine learning model learns the training data too precisely, capturing noise and specific examples rather than general patterns, leading to poor performance

# View Log File

In [10]:
# View the full experiment log as a table
# WHAT THIS CELL DOES:
# Reads your CSV log and displays it as a clean pandas table.
# In a real project you'd use this to compare dozens of runs.

import pandas as pd

df = pd.read_csv("llm_experiment_log.csv")

print(f" Total experiments logged: {len(df)}\n")

# Show key columns in a readable format
display_cols = ["timestamp", "experiment_name", "temperature",
                "total_tokens_used", "reply_score", "score_reason"]

print(df[display_cols].to_string(index=False))

print(f"\n💡 Full replies are saved in the CSV.")
print(f"   Open 'llm_experiment_log.csv' in Excel to read them.")

# Best performing experiment
best = df.loc[df["reply_score"].idxmax()]
print(f"\n🏆 Best experiment: {best['experiment_name']} (Score: {best['reply_score']}/10)")

 Total experiments logged: 3

          timestamp  experiment_name  temperature  total_tokens_used  reply_score           score_reason
2026-06-01 09:09:34    Formal Expert          0.3                517            6   Too brief (17 words)
2026-06-01 09:09:49   Friendly Tutor          0.7                521            7 No clear analogy found
2026-06-01 09:10:01 One-Line Summary          0.0                173            9         One sentence ✅

💡 Full replies are saved in the CSV.
   Open 'llm_experiment_log.csv' in Excel to read them.

🏆 Best experiment: One-Line Summary (Score: 9/10)


### Output Analysis

The word counts (17 and 14 words) tell you the replies were truncated. Gemini 2.5 Flash's thinking tokens consumed most of the 500 token budget before the actual reply could finish. This is a known behaviour of thinking models — they reason internally first, then write the visible reply.
The lesson: with thinking models, always set `max_output_tokens` higher than you think you need. 800-1000 is safer.

Scoring Was Honest
The auto-scorer caught real issues:

- Formal Expert scored 6 — reply was cut off, genuinely too brief
- Friendly Tutor scored 7 — no clear analogy detected in the truncated reply
- One-Line Summary scored 9 — nailed the brief exactly, 30 words, one sentence

This is what an evaluation harness does — removes guesswork and gives you reproducible, comparable scores.

# PROJECT COMPLETE — LLM API EXPLORER
### Full Summary             

## WHAT YOU BUILT

A Python playground that calls the Gemini API, experiments with system prompts, temperatures, roles, and token limits, logs every call to CSV, and evaluates outputs systematically.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## PHASE 2 — First API Call

✅ Stored API key safely using .env + python-dotenv

✅ Created a Gemini client and made your first API call

✅ Read response.text and response.usage_metadata

✅ Understood what max_output_tokens does

✅ Discovered thinking tokens (gemini-2.5-flash reasons internally before writing — those tokens are invisible to you but count against your budget)

---

## PHASE 3 — Roles, System Prompts, Temperature

✅ Same question + different system prompt = completely different tone, depth, and style of answer

✅ Roles explained:

- system → sets AI personality (via system_instruction)
- user → your message (in contents list)
- model → Gemini's reply (used in conversation history)

✅ Temperature behaviour observed:

- 0.0 → deterministic, safe, heavy thinking overhead
- 0.7 → natural, balanced
- 1.5 → creative, varied vocabulary
- 2.0 → unpredictable, more thinking overhead

✅ Multi-turn conversation — you must resend the full history every call. Gemini has no memory by itself. Token cost grows with every turn added.

---

## PHASE 4 — Token Counting Before Sending

✅ Used tiktoken to count tokens BEFORE the API call

✅ Understood context window: 1,000,000 tokens for Gemini 2.5 Flash

✅ Saw how text breaks into tokens

- "Machine learning" = 2 tokens
- "." = its own token

✅ Built estimate_call() — a reusable pre-flight checker

---

## PHASE 5 — Logging System

✅ Built a CSV logger that captures every experiment:

- timestamp
- prompt
- settings
- reply
- tokens
- score

✅ Every call is now reproducible and comparable

✅ This is standard practice in any production AI system

---

## PHASE 6 — Evaluation Harness

✅ Ran 3 experiments on the same question

✅ Auto-scored each reply against defined criteria

✅ Built a summary table using pandas

✅ Identified best performing prompt style: One-Line Summary

✅ Understood that evaluation removes guesswork — you make data-driven decisions about prompts

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## KEY MENTAL MODELS YOU NOW HAVE

### 1. Every AI app = send text in, get text out. Always.

Everything else is just organising that loop better.

### 2. System prompts are the most powerful lever you have.

Before tuning temperature, tune your system prompt.

### 3. Gemini has NO memory.

You build memory by managing conversation history yourself.

### 4. Tokens = money + context.

Count before you send.

### 5. Never trust output quality by feel.

Log and score.